In [20]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix, 
                            roc_auc_score, f1_score, average_precision_score)
import xgboost as xgb
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')



In [21]:
df = pd.read_csv('../data/kaggle_b2_fraud_test_v3.csv')  
print(f"✅ {df.shape[0]} lignes × {df.shape[1]} colonnes")
# print(f"   Taux de fraude: {df['target_is_fraud'].mean():.2%}")


customer_ids = df["customer_id"]


✅ 40000 lignes × 55 colonnes


# 1. Traitement des dates

In [22]:

if 'signup_date' in df.columns:
    df['signup_date'] = pd.to_datetime(df['signup_date'], errors='coerce')
    df['signup_year'] = df['signup_date'].dt.year
    df['signup_month'] = df['signup_date'].dt.month
    df['signup_day_of_week'] = df['signup_date'].dt.dayofweek
    df['days_since_signup'] = (datetime.now() - df['signup_date']).dt.days
    df = df.drop('signup_date', axis=1)
    print("    ✓ Features temporelles créées")



    ✓ Features temporelles créées


# 2. Identification des colonnes

In [23]:


print("\n2️⃣  Identification des colonnes...")
exclude_cols = [
    'customer_id', 'account_id', 'customer_note', 'referrer_code',
    'secondary_email', 'postal_code', 'city', 'last_ticket_subject',
    'post_event_status_code', 'chargeback_resolution_time_days',
    'manual_review_result'
]

# target = 'target_is_fraud'
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in exclude_cols]
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
categorical_cols = [c for c in categorical_cols if c not in exclude_cols]

print(f"    ✓ {len(numeric_cols)} colonnes numériques")
print(f"    ✓ {len(categorical_cols)} colonnes catégorielles")



2️⃣  Identification des colonnes...
    ✓ 33 colonnes numériques
    ✓ 11 colonnes catégorielles


# 3. Valeurs manquantes

In [24]:


# 3. VALEURS MANQUANTES
print("\n3️⃣  Traitement des valeurs manquantes...")
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)

for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna('Unknown', inplace=True)
print("    ✓ Valeurs manquantes traitées")



3️⃣  Traitement des valeurs manquantes...
    ✓ Valeurs manquantes traitées


# 4. Feature engeneering

In [25]:
print("\n4️⃣  Feature Engineering...")
features_created = 0

# Transactions par mois
if 'num_transactions_30d' in df.columns and 'tenure_months' in df.columns:
    df['tx_per_month'] = df['num_transactions_30d'] / (df['tenure_months'] + 1)
    features_created += 1

# Ratio transaction/revenu
if 'avg_amount_30d_eur' in df.columns and 'annual_income_eur' in df.columns:
    df['tx_to_income_ratio'] = df['avg_amount_30d_eur'] / (df['annual_income_eur'] / 12 + 1)
    features_created += 1

# Taux de chargeback
if 'chargebacks_12m' in df.columns and 'num_transactions_30d' in df.columns:
    df['chargeback_rate'] = df['chargebacks_12m'] / (df['num_transactions_30d'] * 4 + 1)
    features_created += 1

# Taux de paiement échoué
if 'failed_payments_6m' in df.columns and 'num_transactions_30d' in df.columns:
    df['failed_payment_rate'] = df['failed_payments_6m'] / (df['num_transactions_30d'] * 2 + 1)
    features_created += 1

# Score de risque composite
risk_cols = [c for c in ['chargebacks_12m', 'failed_payments_6m', 'support_tickets_90d',
             'is_vpn', 'is_new_device'] if c in df.columns]
if len(risk_cols) > 0:
    risk_df = df[risk_cols].copy()
    for col in risk_cols:
        if df[col].std() > 0:
            risk_df[col] = (df[col] - df[col].mean()) / df[col].std()
    df['composite_risk_score'] = risk_df.sum(axis=1)
    features_created += 1

print(f"    ✓ {features_created} nouvelles features créées")



4️⃣  Feature Engineering...
    ✓ 5 nouvelles features créées


# 5. Encodage catégorielles

In [26]:
print("\n5️⃣  Encodage des variables catégorielles...")
label_encoders = {}
for col in categorical_cols:
    if col in df.columns:
        # Réduction cardinalité si > 50 valeurs
        if df[col].nunique() > 50:
            top_values = df[col].value_counts().head(50).index
            df[col] = df[col].apply(lambda x: x if x in top_values else 'Other')
        
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        label_encoders[col] = le

print(f"    ✓ {len(categorical_cols)} colonnes encodées")



5️⃣  Encodage des variables catégorielles...


    ✓ 11 colonnes encodées


# 6. Traitement des outliers

In [27]:
print("\n6️⃣  Traitement des outliers...")
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in exclude_cols ]

for col in numeric_cols:
    if col in df.columns:
        q1 = df[col].quantile(0.01)
        q99 = df[col].quantile(0.99)
        df[col] = df[col].clip(lower=q1, upper=q99)
print(f"    ✓ Outliers traités")




6️⃣  Traitement des outliers...


    ✓ Outliers traités


# 7. Décision finale

In [28]:
features = [c for c in df.columns if c not in exclude_cols ]  # exclut customer_id
df_features = df[features]

# Réinjecter customer_id pour l'export
df_processed = df_features.copy()
if 'customer_id' in df.columns:
    df_processed['customer_id'] = df['customer_id']

# 8. Export

In [29]:

print("\n💾 Export du dataset preprocessé...")
df_processed.to_csv('../data/data_preprocess_rf_test.csv', index=False)
print("✅ data_preprocess_rf_test.csv créé!")


💾 Export du dataset preprocessé...
✅ data_preprocess_rf_test.csv créé!


In [30]:
df_processed

,age,tenure_months,annual_income_eur,credit_score,num_transactions_30d,avg_amount_30d_eur,max_amount_30d_eur,days_since_last_login,support_tickets_90d,chargebacks_12m,...,signup_year,signup_month,signup_day_of_week,days_since_signup,tx_per_month,tx_to_income_ratio,chargeback_rate,failed_payment_rate,composite_risk_score,customer_id
0,43,53,25587.55,639.0,24,106.61,781.55,7.3,0,0,...,2025,9,1,140,0.444444,0.049974,0.0,0.000000,0.136592,CUST_E5RX1BC9II
1,22,5,45378.40,699.0,21,54.17,252.95,4.2,1,0,...,2024,7,2,573,3.500000,0.014321,0.0,0.000000,-1.356902,CUST_BHWIUKERGN
2,42,2,36643.70,765.0,17,44.40,238.79,7.0,1,0,...,2025,2,3,355,5.666667,0.014535,0.0,0.000000,-1.356902,CUST_EXT9NA4CHU
3,39,20,30283.30,573.0,29,38.30,321.44,28.3,0,0,...,2024,8,0,547,1.380952,0.015171,0.0,0.016949,-0.794326,CUST_9FSJE5R1NY
4,54,10,35294.22,624.0,21,70.93,540.48,21.7,0,0,...,2024,2,3,747,1.909091,0.024108,0.0,0.000000,-2.466131,CUST_GDQXMODBED
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39995,44,3,41531.58,622.0,24,42.99,214.60,7.3,0,0,...,2025,12,3,75,6.000000,0.012418,0.0,0.020408,-0.794326,CUST_1OM9UCID91
39996,48,12,36783.80,697.0,25,21.17,69.19,3.8,1,0,...,2024,6,6,618,1.923077,0.006904,0.0,0.000000,-1.356902,CUST_VDEY72BIZP
39997,49,28,55963.60,658.0,18,180.58,410.17,3.6,2,0,...,2024,11,2,468,0.620690,0.038713,0.0,0.000000,-0.247672,CUST_UQEZ9KKIFG
39998,31,7,9638.61,683.0,19,129.93,643.13,13.9,0,0,...,2024,6,5,612,2.375000,0.125772,0.0,0.000000,-2.466131,CUST_IXX0BE9VQD
